In [1]:
import sys
from pathlib import Path
import json
import pickle

ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    repo_root = ROOT
elif (ROOT.parent / 'src').exists():
    repo_root = ROOT.parent
else:
    repo_root = ROOT.parent.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Using repo root: {repo_root}")

Using repo root: D:\git projects\certified-attribution-medical-imaging


In [2]:
# Configuration
import torch
from src.certify.eval.robustness import RobustnessEvaluator

dataset_name = 'isic'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Paths to certification results
cert_results_dir = repo_root / 'notebooks/output/certifications/isic'
output_base = repo_root / 'notebooks/output/eval/robustness/grid/isic'
checkpoint_base = repo_root / 'notebooks/output/checkpoints/isic'

print(f"Cert results dir: {cert_results_dir}")
print(f"Output base: {output_base}")

Device: cpu
Cert results dir: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic
Output base: D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\robustness\grid\isic


In [3]:
# Find all certification results in the directory
import glob

cert_files = sorted(glob.glob(str(cert_results_dir / '**/*.pkl'), recursive=True))
print(f"Found {len(cert_files)} certification result(s):")
for f in cert_files:
    print(f"  {Path(f).name}")

if not cert_files:
    raise FileNotFoundError(f"No certification results found in {cert_results_dir}")

cert_pkl = cert_files[0]
print(f"\nUsing: {cert_pkl}")

Found 2 certification result(s):
  results_20251229_163021.pkl
  results_partial.pkl

Using: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic\results_20251229_163021.pkl


In [4]:
# Load certification results to see available models
with open(cert_pkl, 'rb') as f:
    cert_results = pickle.load(f)

models_in_results = [m for m in cert_results.keys() if cert_results[m]]
print(f"Models with results: {models_in_results}")

for model_name in models_in_results:
    methods = list(cert_results[model_name].keys())
    print(f"  {model_name}: {methods}")

Models with results: ['densenet121']
  densenet121: ['IntegratedGradients', 'GradCAM', 'RISE', 'Occlusion', 'LRP']


In [10]:
# Run robustness evaluation for each model
from collections import defaultdict

all_results = {}

for model_name in models_in_results:
    print(f"\n{'='*60}")
    print(f"Evaluating {model_name}")
    print(f"{'='*60}")
    
    loc_output_dir = output_base / model_name
    loc_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Initialize evaluator
    evaluator = RobustnessEvaluator(
        dataset_name=dataset_name,
        model_name=model_name,
        checkpoint_dir=checkpoint_base,
        device=device,
    )
    
    # Run robustness evaluation
    rob_results_raw = evaluator.evaluate_batch(
        cert_results_pkl=cert_pkl,
        dataset=None,
        output_dir=loc_output_dir,
    )
    
    # Extract results for this model (evaluate_batch returns {model -> {...}})
    rob_results = rob_results_raw.get(model_name, {})
    
    # Save results
    evaluator.save_results_json(rob_results, loc_output_dir / 'robustness_results.json')
    
    # Store for summary and custom visualization
    all_results[model_name] = rob_results
    
    print(f"\nResults saved to: {loc_output_dir}")
    print(f"Methods: {list(rob_results.keys())}")
    print(f"K values: {list(rob_results[list(rob_results.keys())[0]].keys()) if rob_results else 'N/A'}")

print(f"\n{'='*60}")
print("Data collected - generating custom robustness visualizations")
print(f"{'='*60}")


Evaluating densenet121
  Loading checkpoint: D:\git projects\certified-attribution-medical-imaging\notebooks\output\checkpoints\isic\densenet121\final_model.pt
  ✓ Saved results to D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\robustness\grid\isic\densenet121\robustness_results.json

Results saved to: D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\robustness\grid\isic\densenet121
Methods: ['GradCAM', 'IntegratedGradients', 'LRP', 'Occlusion', 'RISE']
K values: [5, 25, 50]

Data collected - generating custom robustness visualizations


In [20]:
# Generate stacked bar chart visualization (Figure 5 style)
import importlib
from src.certify.eval import robustness
importlib.reload(robustness)
from src.certify.eval.robustness import RobustnessEvaluator

for model_name in models_in_results:
    loc_output_dir = output_base / model_name
    
    # Initialize evaluator to use its plot_stacked_certification method
    evaluator = RobustnessEvaluator(
        dataset_name=dataset_name,
        model_name=model_name,
        checkpoint_dir=checkpoint_base,
        device=device,
    )
    
    # Call stacked visualization with results from this model only
    results_for_plot = {model_name: all_results[model_name]}
    evaluator.plot_stacked_certification(results_for_plot, loc_output_dir / 'figures', model_name)

print("\n✓ Stacked bar chart visualization complete")

  Loading checkpoint: D:\git projects\certified-attribution-medical-imaging\notebooks\output\checkpoints\isic\densenet121\final_model.pt


d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  ✓ Saved stacked robustness figure to D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\robustness\grid\isic\densenet121\figures\robustness_stacked.png

✓ Stacked bar chart visualization complete


In [18]:
# Display robustness results summary
import pandas as pd

for model_name in models_in_results:
    print(f"\n{'='*60}")
    print(f"{model_name.upper()}")
    print(f"{'='*60}")
    
    results = all_results[model_name]
    
    for method_name in sorted(results.keys()):
        print(f"\n{method_name}:")
        method_results = results[method_name]
        
        # Create table
        rows = []
        for k in sorted(method_results.keys()):
            metrics = method_results[k]
            rows.append({
                'K%': k,
                'Mean %certified': f"{metrics.get('mean_pct_certified', 0):.2f}%",
                'Std %certified': f"{metrics.get('std_pct_certified', 0):.2f}%",
                'Num images': metrics.get('num_images', 0),
            })
        
        df = pd.DataFrame(rows)
        print(df.to_string(index=False))


DENSENET121

GradCAM:
 K% Mean %certified Std %certified  Num images
  5           0.00%          0.00%           3
 25           0.00%          0.00%           3
 50           0.00%          0.00%           3

IntegratedGradients:
 K% Mean %certified Std %certified  Num images
  5           0.00%          0.00%           3
 25           0.00%          0.00%           3
 50           0.00%          0.00%           3

LRP:
 K% Mean %certified Std %certified  Num images
  5           0.00%          0.00%           2
 25           0.00%          0.00%           2
 50           0.00%          0.00%           2

Occlusion:
 K% Mean %certified Std %certified  Num images
  5           0.00%          0.00%           2
 25           0.00%          0.00%           3
 50           0.00%          0.00%           3

RISE:
 K% Mean %certified Std %certified  Num images
  5           0.00%          0.00%           3
 25           0.00%          0.00%           3
 50           0.00%          0.00%   

In [17]:
# Summary statistics
print(f"\n{'='*60}")
print("ROBUSTNESS EVALUATION COMPLETE")
print(f"{'='*60}")
print(f"\nResults saved to: {output_base}")
print(f"\nStructure:")
for model_name in models_in_results:
    model_dir = output_base / model_name
    print(f"  {model_name}/")
    print(f"    ├── robustness_results.json")
    print(f"    └── figures/")
    print(f"        └── robustness_k*.png")


ROBUSTNESS EVALUATION COMPLETE

Results saved to: D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\robustness\grid\isic

Structure:
  densenet121/
    ├── robustness_results.json
    └── figures/
        └── robustness_k*.png
